# 29 — Probabilidad de texto combinada: logística + BETO

**Idea propia.** En la ablación de la 28, sacar el texto es lo que más le cuesta al modelo en vendedores sin historial (−0,023). La palanca es la **calidad de la probabilidad de texto**. Se usan las probabilidades ya guardadas de BETO (Fase Q, 15 ajustes, `.cache_laboratorio/bert/`): **no se reentrena BETO**.

**Partes:**
1. Cuánto se parecen la logística de texto y BETO: correlación, % de discrepancia, accuracy de cada una en vendedores sin historial.
2. Probabilidad de texto combinada, en lugar de `prob_texto_new`: (a) promedio simple; (b) promedio de logits; (c) peso elegido con los 4 folds del train y evaluado en el quinto; (d) las dos por separado + su diferencia.
3. Medición en el HistGB P7a + duración (comparación pareada contra 0,9412) y, si pasa, en el promedio final (0,9420). Accuracy total y en vendedores sin historial.

**Compatibilidad del cross-fitting** (fold externo k: train = los otros 4 folds, validación = fold k):

| | Filas de train | Filas de validación |
|---|---|---|
| Logística (`PrediccionOOF`) | OOF de una CV interna de 5 folds sobre el train del fold: no vio la fila ni la validación | modelo con todo el train del fold |
| BETO | fila del fold j ← `sin_j_k` (sin los folds j y k): no vio la fila ni la validación | `sin_k` (sin el fold k) |

Las dos cumplen la misma regla (ninguna probabilidad de una fila de train sale de un modelo que la vio, y ninguna vio la validación), así que combinarlas no crea leakage. Que las particiones internas sean distintas no importa.

**Semilla 7: no se puede sin reentrenar BETO.** Los 15 ajustes corresponden a los folds de la semilla 42; con otros folds, la probabilidad de BETO de las filas de train saldría de modelos que vieron la validación.

**Material:**
- promedio de probabilidades (voting *soft*): SIA `S8 - Ensamble.pdf`, slides 6-7 y 24-25;
- logit = log-odds: `material/Regresion Logistica/Presentación_Logística.pdf`, slide 34;
- *log-loss* / entropía cruzada (para elegir el peso de c): `material/Arboles/Árboles.pdf`, slides 12-13 y 32;
- BETO y el fine-tuning: ver Fase Q (no está en el material más allá de SIA S12).

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from joblib import Memory, Parallel, delayed
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      PrediccionOOF, armar_modelo_texto, armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 200)
SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
n = len(crudo)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(n, dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1
carpeta_lab = ".cache_laboratorio"
carpeta_bert = os.path.join(carpeta_lab, "bert")
memoria_texto = Memory(location=carpeta_lab, verbose=0)

oof_ref = pd.read_csv(os.path.join(carpeta_lab, "oof_p7c_duracion.csv"))
assert (oof_ref["fold"].to_numpy() == folds).all()
p_hgb, p_cat = oof_ref["histgb"].to_numpy(), oof_ref["catboost"].to_numpy()
p_final = 0.5 * p_hgb + 0.5 * p_cat


def acc_por_fold(prob, filas=None):
    filas = np.ones(n, bool) if filas is None else filas
    acierto = ((np.asarray(prob) >= 0.5).astype(int) == y_np)
    return pd.Series(acierto[filas]).groupby(folds[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


sin_historial = np.zeros(n, bool)
for k in range(1, 6):
    va = folds == k
    sin_historial[va] = ~train.loc[va, "seller_id"].isin(set(train.loc[~va, "seller_id"])).to_numpy()
acc_hgb, acc_final = acc_por_fold(p_hgb), acc_por_fold(p_final)

# BETO (Fase Q): OOF de los 5 globales y los 10 "de a pares"
bert_global = np.full(n, np.nan)
for k in range(1, 6):
    d = np.load(os.path.join(carpeta_bert, f"sin_{k}.npz"))
    es_train = d["filas"] < n
    bert_global[d["filas"][es_train]] = d["prob"][es_train]
bert_pares = {}
for j in range(1, 6):
    for k in range(j + 1, 6):
        d = np.load(os.path.join(carpeta_bert, f"sin_{j}_{k}.npz"))
        p = np.full(n, np.nan)
        p[d["filas"]] = d["prob"]
        bert_pares[(j, k)] = bert_pares[(k, j)] = p
assert not np.isnan(bert_global).any()


def columna_bert(k):
    col = bert_global.copy()                             # validación (fold k): sin_k
    for j in range(1, 6):
        if j != k:
            col[folds == j] = bert_pares[(j, k)][folds == j]   # train del fold j: sin_j_k
    return col


prob_log = pd.read_csv(os.path.join(carpeta_lab, "oof_texto_logistica.csv"))["prob"].to_numpy()   # OOF con los mismos folds
print("Modelo final:", acc_final.mean().round(4), "| HistGB P7a + duración:", acc_hgb.mean().round(4),
      f"| sin historial: {sin_historial.mean():.1%} de las filas")

Modelo final: 0.942 | HistGB P7a + duración: 0.9412 | sin historial: 30.7% de las filas


## 1. ¿Cuánto se parecen la logística y BETO?

Con las OOF de los dos modelos de texto **solos** (cada fila predicha por un modelo que no la vio, con los mismos 5 folds): correlación, % de filas donde uno dice `new` y el otro `used`, y accuracy de cada uno (y de su promedio) en total y en vendedores sin historial. En las filas donde discrepan, ¿quién acierta más?

In [2]:
discrepan = (prob_log >= 0.5) != (bert_global >= 0.5)
tabla = {}
for nombre, p in {"logística": prob_log, "BETO": bert_global, "promedio": (prob_log + bert_global) / 2}.items():
    tabla[nombre] = {"total": acc_por_fold(p).mean(), "sin historial": acc_por_fold(p, sin_historial).mean(),
                     "con historial": acc_por_fold(p, ~sin_historial).mean(),
                     "donde discrepan": ((p[discrepan] >= 0.5) == y_np[discrepan]).mean()}
print("Correlación:", np.corrcoef(prob_log, bert_global)[0, 1].round(3),
      "| en sin historial:", np.corrcoef(prob_log[sin_historial], bert_global[sin_historial])[0, 1].round(3))
print(f"Discrepan: {discrepan.mean():.1%} del total, {discrepan[sin_historial].mean():.1%} en sin historial")
pd.DataFrame(tabla).T.round(4)

Correlación: 0.889 | en sin historial: 0.843
Discrepan: 11.7% del total, 15.9% en sin historial


,total,sin historial,con historial,donde discrepan
logística,0.8414,0.7835,0.8672,0.4645
BETO,0.8497,0.7926,0.8751,0.5355
promedio,0.8589,0.8036,0.8835,0.6139


## 2. Armado por fold y control

**Qué hace `columnas_texto(k)`:** para el fold externo k devuelve las dos probabilidades en las 70.000 filas:
- logística: se corre el **mismo** `PrediccionOOF(armar_modelo_texto(), random_state=42)` que tiene el `Pipeline`, sobre el train del fold (`fit_transform` → OOF interna) y sobre la validación (`transform` → modelo con todo el train);
- BETO: `columna_bert(k)`.

**Qué hace `armar(modelo, columnas)`:** el pipeline de siempre, pero el transformer `texto` pasa a ser `passthrough` de la primera columna (misma posición: `INDICES_CAT` de CatBoost no cambia), y las columnas extra van al final.

**Control:** con la logística sola como columna, el HistGB tiene que dar **exactamente** las OOF guardadas de P7a. Si da igual, calcular la logística afuera es equivalente a calcularla adentro.

In [3]:
def columnas_texto(k):
    entrena, valida = folds != k, folds == k
    oof = Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)), ("identidad", "passthrough")],
                   memory=memoria_texto)
    columnas = ["title", "warranty_texto"]
    L = np.empty(n)
    L[entrena] = oof.fit_transform(train.loc[entrena, columnas], y[entrena])[:, 0]
    L[valida] = oof.transform(train.loc[valida, columnas])[:, 0]
    return L, columna_bert(k)


inicio = time.time()
texto_por_fold = dict(zip(range(1, 6), Parallel(n_jobs=5)(delayed(columnas_texto)(k) for k in range(1, 6))))
print(f"Logística por fold: {(time.time() - inicio) / 60:.1f} min")
# La logística de validación tiene que coincidir con la OOF guardada (mismo modelo, mismo train)
for k in range(1, 6):
    assert np.allclose(texto_por_fold[k][0][folds == k], prob_log[folds == k])


def armar(modelo, columnas):
    pipeline = armar_pipeline_histgb_combos() if modelo == "histgb" else armar_pipeline_final()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", "passthrough", columnas[:1])
    if len(columnas) > 1:
        transformers.append(("texto_extra", "passthrough", columnas[1:]))
    return pipeline


def correr_fold(k, modelo, variante):
    entrena, valida = folds != k, folds == k
    X = train.copy()
    for nombre, valores in variante(*texto_por_fold[k], entrena).items():
        X[nombre] = valores
    params = {"modelo__cat_features": INDICES_CAT} if modelo == "catboost" else {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = armar(modelo, list(variante(*texto_por_fold[k], entrena))).fit(X[entrena], y[entrena], **params)
    return k, pipeline.predict_proba(X[valida])[:, 1]


def oof(modelo, variante):
    prob = np.empty(n)
    for k, p in Parallel(n_jobs=5)(delayed(correr_fold)(k, modelo, variante) for k in range(1, 6)):
        prob[folds == k] = p
    return prob


control = oof("histgb", lambda L, B, entrena: {"t_log": L})
print("Control: máx |diferencia| con las OOF guardadas de P7a =", np.abs(control - p_hgb).max(),
      "| accuracy", acc_por_fold(control).mean().round(4), "contra", acc_hgb.mean().round(4))

Logística por fold: 0.2 min


Control: máx |diferencia| con las OOF guardadas de P7a = 1.1102230246251565e-16 | accuracy 0.9412 contra 0.9412


## 3. Variantes en el HistGB P7a + duración

- **(a) promedio:** (L + B) / 2.
- **(b) promedio de logits:** σ((logit L + logit B) / 2), con las probabilidades recortadas a [1e-6, 1 − 1e-6] para que el logit no sea infinito.
- **(c) peso elegido:** w·B + (1 − w)·L, con w en 0; 0,05; …; 1 elegido minimizando la *log-loss* sobre las **filas de train del fold** (los otros 4 folds) y aplicado a todo. Se reporta el w de cada fold.
- **(d) separadas + diferencia:** L, B y B − L.

Cada una se compara fold por fold contra el HistGB P7a (0,9412), con la accuracy en vendedores sin historial.

In [4]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def log_loss(p, y_):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -np.mean(y_ * np.log(p) + (1 - y_) * np.log(1 - p))


GRILLA_W = np.round(np.arange(0, 1.001, 0.05), 2)
pesos_elegidos = {}


def peso_elegido(L, B, entrena):
    w = min(GRILLA_W, key=lambda w: log_loss(w * B[entrena] + (1 - w) * L[entrena], y_np[entrena]))
    return {"t_comb": w * B + (1 - w) * L}


VARIANTES = {
    "(a) promedio simple": lambda L, B, entrena: {"t_comb": (L + B) / 2},
    "(b) promedio de logits": lambda L, B, entrena: {"t_comb": 1 / (1 + np.exp(-(logit(L) + logit(B)) / 2))},
    "(c) peso elegido (4 folds)": peso_elegido,
    "(d) separadas + diferencia": lambda L, B, entrena: {"t_log": L, "t_bert": B, "t_dif": B - L},
}
for k in range(1, 6):
    L, B = texto_por_fold[k]
    entrena = folds != k
    pesos_elegidos[k] = min(GRILLA_W, key=lambda w: log_loss(w * B[entrena] + (1 - w) * L[entrena], y_np[entrena]))
print("Peso de BETO elegido en cada fold (c):", pesos_elegidos)

resultados, probs = {}, {}
for nombre, variante in VARIANTES.items():
    inicio = time.time()
    probs[nombre] = oof("histgb", variante)
    a, a_sh, r_sh = acc_por_fold(probs[nombre]), acc_por_fold(probs[nombre], sin_historial), acc_por_fold(p_hgb, sin_historial)
    resultados[nombre] = {**comparar(a, acc_hgb), "sin_hist_antes": r_sh.mean(), "sin_hist_despues": a_sh.mean(),
                          "sin_hist_folds_gana": int((np.round(a_sh - r_sh, 10) > 0).sum()), "minutos": (time.time() - inicio) / 60}
    r = resultados[nombre]
    print(f"{nombre:28s} {r['media']:.4f} dif {r['dif_media']:+.5f} ({r['folds_gana']}/5, desvío {r['dif_desvio']:.5f})"
          f"{' PASA' if r['pasa'] else ''} | sin historial {r['sin_hist_antes']:.4f} -> {r['sin_hist_despues']:.4f}")
pd.DataFrame(resultados).T.round(5)

Peso de BETO elegido en cada fold (c): {1: np.float64(0.6), 2: np.float64(0.6), 3: np.float64(0.6), 4: np.float64(0.6), 5: np.float64(0.6)}


(a) promedio simple          0.9425 dif +0.00127 (5/5, desvío 0.00104) PASA | sin historial 0.8923 -> 0.8948


(b) promedio de logits       0.9421 dif +0.00087 (5/5, desvío 0.00094) PASA | sin historial 0.8923 -> 0.8941


(c) peso elegido (4 folds)   0.9427 dif +0.00146 (5/5, desvío 0.00103) PASA | sin historial 0.8923 -> 0.8950


(d) separadas + diferencia   0.9423 dif +0.00104 (4/5, desvío 0.00094) PASA | sin historial 0.8923 -> 0.8933


,media,dif_media,dif_desvio,folds_gana,pasa,sin_hist_antes,sin_hist_despues,sin_hist_folds_gana,minutos
(a) promedio simple,0.942514,0.001271,0.001037,5,True,0.892298,0.894836,5,0.474154
(b) promedio de logits,0.942114,0.000871,0.000945,5,True,0.892298,0.894118,4,0.509105
(c) peso elegido (4 folds),0.9427,0.001457,0.001032,5,True,0.892298,0.895046,3,0.506255
(d) separadas + diferencia,0.942286,0.001043,0.000935,4,True,0.892298,0.893334,4,0.490761


## 4. Si alguna pasa: en el promedio final (CatBoost + HistGB)

Se rearma también el CatBoost con la misma variante en lugar de `prob_texto_new` y se compara el promedio 50/50 contra el modelo final (0,9420). La confirmación con la semilla 7 requeriría 15 ajustes nuevos de BETO (~4,4 h): queda para decidir aparte.

In [5]:
pasan = [v for v in VARIANTES if resultados[v]["pasa"]]
print("Pasan en el HistGB:", pasan or "ninguna")
for nombre in pasan:
    inicio = time.time()
    p_cat_v = oof("catboost", VARIANTES[nombre])
    prom = 0.5 * p_cat_v + 0.5 * probs[nombre]
    a_sh, r_sh = acc_por_fold(prom, sin_historial), acc_por_fold(p_final, sin_historial)
    resultados[f"PROMEDIO FINAL {nombre} (vs 0,9420)"] = {**comparar(acc_por_fold(prom), acc_final),
                                                            "sin_hist_antes": r_sh.mean(), "sin_hist_despues": a_sh.mean(),
                                                            "minutos": (time.time() - inicio) / 60}
    probs[f"catboost {nombre}"] = p_cat_v
    print(nombre, {k: round(float(v), 5) for k, v in resultados[f"PROMEDIO FINAL {nombre} (vs 0,9420)"].items()})
pd.DataFrame({"fold": folds, "y": y_np, **probs}).to_csv(os.path.join(carpeta_lab, "oof_texto_combinado.csv"), index=False)
pd.DataFrame(resultados).T.to_csv(os.path.join(carpeta_lab, "texto_combinado.csv"))
pd.DataFrame(resultados).T.round(5)

Pasan en el HistGB: ['(a) promedio simple', '(b) promedio de logits', '(c) peso elegido (4 folds)', '(d) separadas + diferencia']


(a) promedio simple {'media': 0.94353, 'dif_media': 0.00153, 'dif_desvio': 0.0011, 'folds_gana': 5.0, 'pasa': 1.0, 'sin_hist_antes': 0.89306, 'sin_hist_despues': 0.8962, 'minutos': 1.58286}


(b) promedio de logits {'media': 0.94371, 'dif_media': 0.00171, 'dif_desvio': 0.00128, 'folds_gana': 5.0, 'pasa': 1.0, 'sin_hist_antes': 0.89306, 'sin_hist_despues': 0.8962, 'minutos': 1.63708}


(c) peso elegido (4 folds) {'media': 0.94354, 'dif_media': 0.00154, 'dif_desvio': 0.00139, 'folds_gana': 4.0, 'pasa': 1.0, 'sin_hist_antes': 0.89306, 'sin_hist_despues': 0.89577, 'minutos': 1.72304}


(d) separadas + diferencia {'media': 0.94344, 'dif_media': 0.00144, 'dif_desvio': 0.00085, 'folds_gana': 4.0, 'pasa': 1.0, 'sin_hist_antes': 0.89306, 'sin_hist_despues': 0.89519, 'minutos': 1.58666}


,media,dif_media,dif_desvio,folds_gana,pasa,sin_hist_antes,sin_hist_despues,sin_hist_folds_gana,minutos
(a) promedio simple,0.942514,0.001271,0.001037,5,True,0.892298,0.894836,5,0.474154
(b) promedio de logits,0.942114,0.000871,0.000945,5,True,0.892298,0.894118,4,0.509105
(c) peso elegido (4 folds),0.9427,0.001457,0.001032,5,True,0.892298,0.895046,3,0.506255
(d) separadas + diferencia,0.942286,0.001043,0.000935,4,True,0.892298,0.893334,4,0.490761
"PROMEDIO FINAL (a) promedio simple (vs 0,9420)",0.943529,0.001529,0.001098,5,True,0.893057,0.896197,NaN,1.582858
"PROMEDIO FINAL (b) promedio de logits (vs 0,9420)",0.943714,0.001714,0.001283,5,True,0.893057,0.896201,NaN,1.637075
"PROMEDIO FINAL (c) peso elegido (4 folds) (vs 0,9420)",0.943543,0.001543,0.001394,4,True,0.893057,0.89577,NaN,1.723035
"PROMEDIO FINAL (d) separadas + diferencia (vs 0,9420)",0.943443,0.001443,0.000854,4,True,0.893057,0.895189,NaN,1.586664
